In [1]:
import sys
from pathlib import Path

# Get project root
project_root = Path("..").resolve()

# Add root to sys.path
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("PYTHONPATH updated with:", project_root)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.models import resnet18

from src.dataset import TrickSkiDataset

# --------------------------------------
# DEFINE DIRS (needed after restart)
# --------------------------------------
DATA_DIR = project_root / "trickski_data"
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR  = DATA_DIR / "test"

print("Train dir:", TRAIN_DIR)
print("Test dir:", TEST_DIR)

# --------------------------------------
# DEVICE
# --------------------------------------
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print("Using device:", DEVICE)

# --------------------------------------
# SIMPLE TRANSFORMS — NO CROP, NO AUGMENTATION
# --------------------------------------
# train_transform = transforms.Compose([
#     transforms.Resize((224, 224)),   # direct resize only
#     transforms.ToTensor(),
#     transforms.Normalize(
#         mean=[0.485, 0.456, 0.406],
#         std=[0.229, 0.224, 0.225]
#     ),
# ])

# test_transform = transforms.Compose([
#     transforms.Resize((224, 224)),
#     transforms.ToTensor(),
#     transforms.Normalize(
#         mean=[0.485, 0.456, 0.406],
#         std=[0.229, 0.224, 0.225]
#     ),
# ])
# --------------------------------------
# *** UPDATED TRANSFORMS — NO FLIP, NO ROTATION, NO COLOR JITTER ***
# SAFE FOR ORIENTATION CLASSIFICATION
# --------------------------------------
train_transform = transforms.Compose([
    transforms.Resize(256),        # keeps aspect ratio, enlarges image
    transforms.CenterCrop(224),    # focuses on skier (usually central)
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

# --------------------------------------
# LOAD DATASETS
# --------------------------------------
BATCH_SIZE = 8
train_dataset = TrickSkiDataset(TRAIN_DIR, transform=train_transform)
test_dataset  = TrickSkiDataset(TEST_DIR,  transform=test_transform)

print("Train classes:", train_dataset.class_to_idx)
print("Test classes:", test_dataset.class_to_idx)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Dataset sizes:", len(train_dataset), len(test_dataset))

# --------------------------------------
# LOAD PRETRAINED RESNET18
# --------------------------------------
model = resnet18(weights="IMAGENET1K_V1")

# Replace final layer for 4 classes
model.fc = nn.Linear(model.fc.in_features, 4)
model = model.to(DEVICE)

# Loss & optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)


PYTHONPATH updated with: /Users/calvindebellis/Desktop/539 Final Project
Train dir: /Users/calvindebellis/Desktop/539 Final Project/trickski_data/train
Test dir: /Users/calvindebellis/Desktop/539 Final Project/trickski_data/test
Using device: mps
Train classes: {'train_back': 0, 'train_forward': 1, 'train_left': 2, 'train_right': 3}
Test classes: {'test_back': 0, 'test_forward': 1, 'test_left': 2, 'test_right': 3}
Dataset sizes: 180 60


In [2]:
from torchvision.models import resnet18

# Create pretrained ResNet18
model = resnet18(weights="IMAGENET1K_V1")

# Replace the classification head for 4 classes
model.fc = nn.Linear(model.fc.in_features, 4)

# Move to CPU/GPU/MPS
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)  # lower LR for pretrained models



In [3]:
NUM_EPOCHS = 40

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0

    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    # --------------------
    # Evaluate on test set
    # --------------------
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = model(imgs)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    test_acc = correct / total

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} - Loss: {avg_loss:.4f} - Test Acc: {test_acc:.4f}")



Epoch 1/40 - Loss: 1.5239 - Test Acc: 0.2167
Epoch 2/40 - Loss: 0.7208 - Test Acc: 0.3000
Epoch 3/40 - Loss: 0.2882 - Test Acc: 0.3667
Epoch 4/40 - Loss: 0.1099 - Test Acc: 0.4000
Epoch 5/40 - Loss: 0.0778 - Test Acc: 0.4167
Epoch 6/40 - Loss: 0.0663 - Test Acc: 0.5000
Epoch 7/40 - Loss: 0.0647 - Test Acc: 0.4167
Epoch 8/40 - Loss: 0.1492 - Test Acc: 0.3333
Epoch 9/40 - Loss: 0.2555 - Test Acc: 0.3333
Epoch 10/40 - Loss: 0.2066 - Test Acc: 0.4500
Epoch 11/40 - Loss: 0.1209 - Test Acc: 0.6000
Epoch 12/40 - Loss: 0.0487 - Test Acc: 0.6000
Epoch 13/40 - Loss: 0.0810 - Test Acc: 0.5500
Epoch 14/40 - Loss: 0.0618 - Test Acc: 0.6833
Epoch 15/40 - Loss: 0.0759 - Test Acc: 0.7000
Epoch 16/40 - Loss: 0.0576 - Test Acc: 0.5833
Epoch 17/40 - Loss: 0.0558 - Test Acc: 0.7333
Epoch 18/40 - Loss: 0.0354 - Test Acc: 0.7500
Epoch 19/40 - Loss: 0.0250 - Test Acc: 0.5667
Epoch 20/40 - Loss: 0.0513 - Test Acc: 0.7000
Epoch 21/40 - Loss: 0.0299 - Test Acc: 0.6500
Epoch 22/40 - Loss: 0.0775 - Test Acc: 0.75

In [4]:
model.eval()
correct, total = 0, 0

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        outs = model(imgs)
        _, preds = torch.max(outs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total * 100
accuracy


81.66666666666667